# F4-multivar-calculus — Session 1: Partial Derivatives and the Gradient

*One class session, roughly 80 minutes. Prerequisite: F2-vectors (vectors,
norms, dot products, unit vectors) and, through it, F1-scientific-python
(broadcasting, axis aggregations, seeded randomness, matplotlib).*

**This session:** functions that take several inputs at once, the partial
derivative as an ordinary Calc AB derivative taken along one axis with
every other input frozen, the gradient as the vector that collects all the
partials, the direction the gradient points (the single most useful fact in
this unit), and how to verify any hand-derived formula numerically with
central differences.

Try every checkpoint by hand first, then verify with NumPy.
Answers are collected at the end of this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Functions with Several Inputs

**Motivation.**
Every function in Calc AB takes one number in and gives one number out.
Most quantities you actually care about depend on several numbers at once:
the signal strength a drone measures depends on *where it is* — an $x$ and
a $y$; the pressure in a room depends on three coordinates; the mismatch
score in Session 2 will depend on every coefficient of a prediction rule
simultaneously.
This unit is about doing calculus when the input is a vector.

**Definition.**
A **function of several variables** assigns one output number to each
combination of input numbers: $f(x, y)$, $g(u, v, w)$, or in general
$f(w)$ where $w$ is a vector with $d$ entries.
Nothing else changes — plug the inputs in, get one number out.

**Two pictures for two inputs.**
The graph of $z = f(x, y)$ is a *surface* floating over the $xy$-plane.
The top-down view of that surface is a **contour map**: each curve joins
the input points where $f$ takes one fixed value, exactly like elevation
lines on a hiking map.
Contour lines packed close together mean the surface is steep there; lines
far apart mean it is nearly flat.

**Worked example.**
Take $f(x, y) = x^2 y + 3y$ — the running example for this whole session.
By hand: $f(2, 1) = 4 + 3 = 7$, $\;f(0, 5) = 0 + 15 = 15$,
$\;f(-2, 1) = 4 + 3 = 7$ (the square erases the sign of $x$).
Below we evaluate it and draw its contour map.
To evaluate $f$ on a whole *grid* of inputs at once, we use the F1
broadcasting rules: a $(1, 301)$ row of $x$ values against a $(201, 1)$
column of $y$ values fills a $(201, 301)$ table of outputs — rows walking
along $y$, columns along $x$.

In [ ]:
def f(x, y):
    return x**2 * y + 3 * y


print("f(2, 1)  =", f(2.0, 1.0))    # 4 + 3 = 7
print("f(0, 5)  =", f(0.0, 5.0))    # 15
print("f(-2, 1) =", f(-2.0, 1.0))   # 7 again: the square erases the sign

In [ ]:
xs = np.linspace(-3, 3, 301)          # shape (301,)
ys = np.linspace(-2, 2, 201)          # shape (201,)
F = f(xs[None, :], ys[:, None])       # (1, 301) with (201, 1) -> (201, 301)

plt.figure(figsize=(6, 4.5))
level_set = plt.contour(xs, ys, F, levels=12)
plt.clabel(level_set, inline=True, fontsize=7)
plt.xlabel("x")
plt.ylabel("y")
plt.title(r"Contour map of $f(x, y) = x^2 y + 3y$")
plt.show()

### Checkpoint 1

1. By hand: for $g(u, v) = u v^2 - 2u$, compute $g(3, 2)$ and $g(-1, 4)$.
2. Without running code: `xs[None, :]` has shape `(1, 301)` and
   `ys[:, None]` has shape `(201, 1)`. What is the shape of the value
   table `F`, and which axis (rows or columns) walks along $x$?
3. On a contour map, what does it mean about the surface when the contour
   lines bunch tightly together?

## 2. Partial Derivatives: Freeze, Then Differentiate

**Motivation.**
"What is the slope of $f$ at this point?" is ambiguous on a surface — the
slope depends on which way you walk.
The simplest directions to try first are the axis directions: how fast does
$f$ change if *only* $x$ moves, or *only* $y$?

**The slice story.**
Freeze $y$ at a value $b$.
Then $x \mapsto f(x, b)$ is an ordinary one-variable function — a vertical
*slice* of the surface — and it has an ordinary Calc AB derivative.
That derivative is the **partial derivative of $f$ with respect to $x$**.
Freezing $x$ instead and sliding $y$ gives the partial with respect to
$y$.
A partial derivative is not a new kind of derivative; it is the old
derivative applied to a slice.

**Definition.**
$$\frac{\partial f}{\partial x}(x, y)
  = \lim_{h \to 0} \frac{f(x + h,\, y) - f(x, y)}{h},
\qquad
\frac{\partial f}{\partial y}(x, y)
  = \lim_{h \to 0} \frac{f(x,\, y + h) - f(x, y)}{h}.$$
The curly $\partial$ ("partial") replaces $d$ to remind you that the other
inputs are frozen.
Shorthand: $f_x$ and $f_y$.
In practice you never touch the limit: **differentiate normally, treating
every other variable as a constant.**

**Worked hand computations.**
For the running example $f(x, y) = x^2 y + 3y$:

- $\dfrac{\partial f}{\partial x}$: with $y$ frozen, $x^2 y$ is
  $(\text{constant}) \cdot x^2$, derivative $2xy$; and $3y$ is a pure
  constant, derivative $0$.
  Total: $f_x = 2xy$.
- $\dfrac{\partial f}{\partial y}$: with $x$ frozen, $x^2 y$ is
  $(\text{constant}) \cdot y$, derivative $x^2$; and $3y$ gives $3$.
  Total: $f_y = x^2 + 3$.

At the point $(2, 1)$: $f_x(2, 1) = 4$ and $f_y(2, 1) = 7$.

One more, with mixed terms — $g(u, v) = u^3 - 2uv + v^2$:
$g_u = 3u^2 - 2v$ (the $v^2$ term dies), and $g_v = -2u + 2v$ (the $u^3$
term dies).
Each partial keeps exactly the terms its variable appears in.

The code below draws the slices of $f$ for several frozen values of $y$:
every slice is an ordinary parabola, and the tangent line on the $y = 1$
slice at $x = 2$ has slope $f_x(2, 1) = 4$.

In [ ]:
slice_xs = np.linspace(-3, 3, 200)

plt.figure(figsize=(6, 4))
for b in [-1.0, 0.0, 1.0, 2.0]:
    plt.plot(slice_xs, f(slice_xs, b), label=f"y frozen at {b}")

# Tangent on the y = 1 slice at x = 2: slope f_x(2, 1) = 2*2*1 = 4.
x0, b = 2.0, 1.0
xt = np.linspace(1.2, 2.8, 2)
plt.plot(xt, f(x0, b) + 4.0 * (xt - x0), "k--", linewidth=2,
         label="tangent at x=2 on the y=1 slice (slope 4)")
plt.xlabel("x")
plt.ylabel("value on the slice")
plt.title("Freezing y turns the surface into ordinary curves")
plt.legend(fontsize=8)
plt.show()

### Checkpoint 2

1. By hand: both partials of $f(x, y) = 5x^2 y^3$.
2. Both partials of $h(s, t) = s^3 + 4st - t^2$, each evaluated at
   $(s, t) = (1, 2)$.
3. True or false, with a one-line reason:
   $\dfrac{\partial}{\partial x}\, y^4 = 0$.

## 3. Every Calc AB Rule Still Works

**Motivation.**
Partial derivatives need no new differentiation rules.
Product rule, quotient rule, chain rule — all apply unchanged, one
variable at a time, with the frozen variables riding along as constants.
This is where most of the exam's hand-computation items live.

**Worked example (chain rule inside a partial).**
$p(a, b) = (a + 2b)^3$.
For $\partial p/\partial a$: the outer function is $(\cdot)^3$, the inner
is $a + 2b$, whose $a$-derivative is $1$:
$$p_a = 3(a + 2b)^2 \cdot 1 .$$
For $\partial p/\partial b$: same outer factor, but the inner
$b$-derivative is $2$:
$$p_b = 3(a + 2b)^2 \cdot 2 .$$
**The inner derivative is not optional** — forgetting that factor $2$ is
the classic error.
At $(a, b) = (1, 1)$: the inner value is $3$, so $p_a = 27$ and
$p_b = 54$.

**Worked example (product rule and chain rule together).**
$q(x, y) = x\, e^{xy}$.

- $\partial q/\partial x$: product rule on $x \cdot e^{xy}$ (both factors
  contain $x$), and the exponent's $x$-derivative is $y$:
  $$q_x = 1 \cdot e^{xy} + x \cdot y\, e^{xy} = (1 + xy)\, e^{xy}.$$
- $\partial q/\partial y$: now the leading $x$ is a frozen constant
  factor, and the exponent's $y$-derivative is $x$:
  $$q_y = x \cdot x\, e^{xy} = x^2 e^{xy}.$$

The code checks both formulas with a tiny symmetric nudge — the central
difference $\big(q(x_0 + h, y_0) - q(x_0 - h, y_0)\big)/(2h)$.
Section 6 studies this checker properly; here it is just a safety net.

In [ ]:
def q(x, y):
    return x * np.exp(x * y)


x0, y0, h = 1.0, 0.5, 1e-6
qx_hand = (1 + x0 * y0) * np.exp(x0 * y0)
qy_hand = x0**2 * np.exp(x0 * y0)
qx_num = (q(x0 + h, y0) - q(x0 - h, y0)) / (2 * h)
qy_num = (q(x0, y0 + h) - q(x0, y0 - h)) / (2 * h)

print("dq/dx  hand:", qx_hand, "  numeric:", qx_num)
print("dq/dy  hand:", qy_hand, "  numeric:", qy_num)
print("gaps:", abs(qx_hand - qx_num), abs(qy_hand - qy_num))

### Checkpoint 3

1. By hand: both partials of $(3u - v)^4$ (mind the inner derivatives).
2. Both partials of $y\, e^{2x}$, evaluated at $(x, y) = (0, 5)$.
3. Both partials of $\sin(xy)$.

## 4. The Gradient: All Partials in One Vector

**Motivation.**
A function of $d$ inputs has $d$ partial derivatives at every point — one
per input.
Carrying them separately is clumsy; packing them into a single vector
turns all of F2's machinery (norms, unit vectors, dot products) loose on
derivatives.

**Definition.**
The **gradient** of $f$ is the vector of all its partials:
$$\nabla f = \left(\frac{\partial f}{\partial x_1},\,
  \frac{\partial f}{\partial x_2},\, \dots,\,
  \frac{\partial f}{\partial x_d}\right).$$
The symbol $\nabla$ is read "grad".
The gradient is a plain F2 vector: it has a norm
$\lVert\nabla f\rVert$, it can be normalized to a unit vector, it can be
dotted with other vectors.
And it *changes from point to point* — you evaluate it where you stand.

**Worked example.**
For the running $f(x, y) = x^2 y + 3y$:
$\nabla f = (2xy,\; x^2 + 3)$.
At $(2, 1)$: $\nabla f = (4, 7)$, with norm $\sqrt{16 + 49} = \sqrt{65}$.
At $(0, 5)$: $\nabla f = (0, 3)$ — a completely different arrow.

**Worked example (indexed-sum form).**
For a vector input $w$ with $d$ entries, let
$f(w) = \sum_{k=0}^{d-1} w_k^2$.
To find $\partial f/\partial w_j$, scan the sum: only the $k = j$ term
contains $w_j$; every other term is a frozen constant.
So
$$\frac{\partial f}{\partial w_j} = 2 w_j
  \qquad\Longrightarrow\qquad \nabla f = 2w .$$
This scan-the-sum move — *only the terms containing $w_j$ survive* — is
the workhorse of Session 2.

The code draws gradient arrows of the running example on its contour map.
Watch what the arrows do: they cross the contours at right angles and
point toward higher values.

In [ ]:
def f_x(x, y):
    return 2 * x * y


def f_y(x, y):
    return x**2 + 3 + 0 * y   # + 0*y so the result broadcasts to the full grid


gx = np.linspace(-2.5, 2.5, 9)
gy = np.linspace(-1.5, 1.5, 7)
U = f_x(gx[None, :], gy[:, None])     # (7, 9) table of df/dx values
V = f_y(gx[None, :], gy[:, None])     # (7, 9) table of df/dy values

plt.figure(figsize=(6.5, 4.5))
plt.contour(xs, ys, F, levels=12, linewidths=0.7)
plt.quiver(gx, gy, U, V, color="C3", width=0.004)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Gradient arrows point uphill, straight across the contours")
plt.show()

### Checkpoint 4

1. Compute $\nabla g$ for $g(x, y) = x^2 + xy$, then evaluate it at
   $(1, 2)$.
2. For $f(w) = \sum_{k=0}^{d-1} 3 w_k$ (a $d$-entry input), what is
   $\nabla f$, and how many entries does it have?
3. From the picture: what angle does a gradient arrow make with the
   contour line through the same point?

## 5. Which Way Is Steepest? The Direction Property

**Motivation.**
A drone measuring signal strength over a field wants the signal to grow as
fast as possible.
It knows the two partials where it currently hovers.
Which compass direction should it fly?
"Along the biggest partial" is the tempting answer — and it is wrong.

**The rate in an arbitrary direction.**
Let $u = (u_x, u_y)$ be a *unit* vector (F2: length exactly 1), and take a
tiny step of length $t$ along $u$.
Then $x$ changes by $t\,u_x$ and $y$ changes by $t\,u_y$.
Each input's nudge moves $f$ by (its partial) $\times$ (its nudge), and
the two contributions add:
$$\Delta f \approx f_x \cdot t\,u_x + f_y \cdot t\,u_y
  = t\,\big(\nabla f \cdot u\big).$$
So the instantaneous rate of change of $f$ *per unit of distance* in the
direction $u$ is the dot product
$$\text{rate along } u \;=\; \nabla f \cdot u .$$
(That "each nudge contributes, and contributions add" step is Session 2's
chain rule in disguise — we earn it properly there.)

**F2 finishes the argument.**
By the geometric form of the dot product,
$$\nabla f \cdot u = \lVert\nabla f\rVert\,\lVert u\rVert \cos\theta
  = \lVert\nabla f\rVert \cos\theta,$$
where $\theta$ is the angle between $u$ and the gradient.
Since $\cos\theta$ peaks at $\theta = 0$:

- **Steepest increase:** $u = \nabla f / \lVert\nabla f\rVert$, with rate
  $\lVert\nabla f\rVert$ — *the gradient points straight uphill, and its
  length IS the uphill rate.*
- **Steepest decrease:** the opposite direction,
  $-\nabla f / \lVert\nabla f\rVert$, rate $-\lVert\nabla f\rVert$.
- **No change:** any $u$ orthogonal to $\nabla f$
  ($\theta = 90^\circ$) — walking along a contour line.

**Worked example.**
Numerical probing tells the drone $\partial S/\partial x = 3$ and
$\partial S/\partial y = -4$ at its current spot, so
$\nabla S = (3, -4)$ with $\lVert\nabla S\rVert = 5$.

- Fastest increase: fly along $u = (3/5,\, -4/5)$ — east and a bit more
  toward south — gaining $5$ signal units per meter.
- Due north, $u = (0, 1)$: rate $= \nabla S \cdot (0, 1) = -4$ — the
  signal *drops*.
- Due east, $u = (1, 0)$: rate $3$.
  East is the best *axis* direction, but the diagonal $(3/5, -4/5)$ beats
  it, $5 > 3$: the biggest single partial does not name the best
  direction, because both partials contribute to a diagonal's rate.

The code sweeps every direction and confirms the maximum sits exactly at
the gradient's own direction, with rate $\lVert\nabla S\rVert = 5$.

In [ ]:
grad = np.array([3.0, -4.0])
theta = np.linspace(0, 2 * np.pi, 721)
rates = grad[0] * np.cos(theta) + grad[1] * np.sin(theta)   # grad . u(theta)

grad_angle = np.arctan2(grad[1], grad[0]) % (2 * np.pi)

plt.figure(figsize=(6, 3.5))
plt.plot(theta, rates)
plt.axhline(0, color="gray", linewidth=0.5)
plt.axvline(grad_angle, color="C3", linestyle="--",
            label="the gradient's own direction")
plt.xlabel("direction angle (radians)")
plt.ylabel("rate of change along u")
plt.title("The rate peaks in the gradient direction, at ||grad|| = 5")
plt.legend()
plt.show()

print("max swept rate :", rates.max())
print("||grad||       :", np.sqrt(np.sum(grad**2)))
print("rate due north :", rates[np.argmin(np.abs(theta - np.pi / 2))])
print("rate due east  :", rates[0])

### Checkpoint 5

1. At some point, $\partial f/\partial x = -6$ and
   $\partial f/\partial y = 8$.
   Give the unit direction of steepest increase and the rate of increase
   along it.
2. Same point: what is the rate of change moving due east, $u = (1, 0)$?
3. Same point: give the two unit directions along which the instantaneous
   rate of change is exactly $0$.

## 6. Trust, but Verify: Central Differences

**Motivation.**
Hand algebra is exactly where sign slips and dropped factors hide, and the
exam's constrained-coding problems ask you to build the antidote yourself:
a *numeric* derivative that checks the formula (that is p05 and p06, and
the checker halves of p07–p10).
The tool is pure F1: evaluate the function a few times and difference.

**Forward vs. central.**
The limit definition suggests the **forward difference**
$\big(f(x + h) - f(x)\big)/h$, whose error shrinks proportionally to $h$.
The **central difference**
$$\frac{f(x + h) - f(x - h)}{2h}$$
straddles the point symmetrically; the bend of the curve cancels between
the two sides, and the error shrinks like $h^2$ — dramatically better for
the same tiny cost.
(A pleasant special case: on any quadratic the cancellation is *perfect* —
the central difference is exact for every $h$.)
Central is the default in this course.

**Partials, numerically.**
To estimate $\partial f/\partial x_j$, nudge **one coordinate at a time**:
add and subtract $h$ in entry $j$ only, holding the rest — the freeze,
performed by the computer.
The full gradient checker repeats this once per coordinate.
A loop over the $d$ coordinates is fine *in a checker*: checkers are
scaffolding, and exam ban lists say explicitly when a checker "MAY
loop".

In [ ]:
def partial_x_num(f, x, y, h=1e-6):
    return (f(x + h, y) - f(x - h, y)) / (2 * h)


def partial_y_num(f, x, y, h=1e-6):
    return (f(x, y + h) - f(x, y - h)) / (2 * h)


# Section 2's hand answers at (2, 1) were (4, 7):
print("df/dx at (2,1):", partial_x_num(f, 2.0, 1.0), "   hand: 4")
print("df/dy at (2,1):", partial_y_num(f, 2.0, 1.0), "   hand: 7")

In [ ]:
def num_gradient(f, point, h=1e-6):
    """Central-difference gradient: nudge ONE coordinate at a time."""
    point = np.asarray(point, dtype=float)
    grad_est = np.zeros_like(point)
    for j in range(point.shape[0]):        # a checker MAY loop
        step = np.zeros_like(point)
        step[j] = h
        grad_est[j] = (f(point + step) - f(point - step)) / (2 * h)
    return grad_est


def sum_sq(w):
    return np.sum(w**2)


w0 = np.array([1.0, -2.0, 0.5])
est = num_gradient(sum_sq, w0)
exact = 2 * w0                             # Section 4's hand result
print("numeric:", est)
print("exact  :", exact)
print("max gap:", np.abs(est - exact).max())

**Choosing $h$.**
Two failure modes squeeze the choice from both sides:

- $h$ **too large**: the secant no longer approximates the tangent — the
  curvature error (the $h^2$ term) dominates.
- $h$ **too small**: $f(x + h)$ and $f(x - h)$ agree in almost all their
  floating-point digits, and subtracting nearly equal floats destroys
  precision — rounding noise dominates (in the extreme the two values are
  the *same* float and the estimate is exactly 0).

The error curve is a valley.
For central differences on well-scaled inputs,
$h \approx 10^{-6}$ to $10^{-5}$ sits near the bottom; that is the course
default.
The sweep below shows the valley for $\partial/\partial x\; e^{xy}$ at
$(2, 1)$ (true value $e^2$):

In [ ]:
def s_fn(x, y):
    return np.exp(x * y)


true_val = np.exp(2.0)              # d/dx e^{xy} = y e^{xy} -> e^2 at (2, 1)
hs = np.logspace(-13, -1, 25)
ests = (s_fn(2.0 + hs, 1.0) - s_fn(2.0 - hs, 1.0)) / (2 * hs)
errs = np.abs(ests - true_val)

plt.figure(figsize=(6, 3.5))
plt.loglog(hs, errs, marker="o", markersize=3)
plt.axvline(1e-6, color="C2", linestyle="--", label="course default h = 1e-6")
plt.xlabel("step size h")
plt.ylabel("absolute error")
plt.title("Central-difference error: rounding noise (left), curvature (right)")
plt.legend()
plt.show()

### Checkpoint 6

1. By hand: the central difference of $f(x) = x^2$ at $x = 3$ with the
   absurdly large $h = 1$ is $\big(f(4) - f(2)\big)/2$.
   Compute it, compare with $f'(3)$, and say what this illustrates.
2. Rank $h \in \{0.5,\; 10^{-6},\; 10^{-13}\}$ from best to worst for a
   central-difference partial of $e^{xy}$, naming each loser's failure
   mode.
3. In `num_gradient`, why must the checker nudge one coordinate at a
   time?
   What quantity would nudging *all* coordinates by $h$ at once estimate
   instead?

## 7. Common Pitfalls I

**Pitfall 1 — forgetting to freeze the other variables.**
Asked for $\partial/\partial x$ of $f(x, y) = x y^2$, a student
differentiates *both* factors — "product rule everywhere" — and claims
$y^2 + 2xy$.
But with $y$ frozen, $x y^2$ is $(\text{constant}) \cdot x$: the partial
is just $y^2$.
The numeric checker exposes the claim instantly:

In [ ]:
def f1(x, y):
    return x * y**2


x0, y0 = 2.0, 3.0
claim_broken = y0**2 + 2 * x0 * y0    # BROKEN: differentiated the frozen y too
claim_fixed = y0**2                   # y frozen: (constant) * x, slope y^2
numeric = partial_x_num(f1, x0, y0)

print("broken claim:", claim_broken)  # 21
print("fixed claim :", claim_fixed)   # 9
print("numeric     :", numeric)       # ~9 -> the fix is right

The product rule is for two factors that *both move*.
Under $\partial/\partial x$, only factors containing $x$ move; everything
else is a constant coefficient along for the ride.

**Pitfall 2 — mixing up which partial is which.**
For the running $f(x, y) = x^2 y + 3y$, a student reports
$\partial f/\partial x = x^2 + 3$.
That is a correct formula — for $\partial f/\partial y$.
The habit that catches it: *the variable named in $\partial/\partial x$ is
the one you nudge; everything else is furniture.*
A quick sanity probe helps too: $f_x = 2xy$ must vanish at $x = 0$, while
the mixed-up $x^2 + 3$ never vanishes.
The same confusion appears in code as an argument-order bug — nudging the
wrong argument computes the wrong partial with no error message:

In [ ]:
x0, y0 = 2.0, 1.0

claim_broken = x0**2 + 3               # BROKEN: this is df/dy, not df/dx
claim_fixed = 2 * x0 * y0              # df/dx = 2xy

numeric_x = partial_x_num(f, x0, y0)   # nudges the FIRST argument
print("claimed df/dx (broken):", claim_broken, "   numeric df/dx:", numeric_x)
print("claimed df/dx (fixed) :", claim_fixed)

# The code version of the same bug: nudging y while calling it df/dx.
wrong_arg = (f(x0, y0 + 1e-6) - f(x0, y0 - 1e-6)) / 2e-6   # BROKEN checker
print("checker that nudges the wrong argument:", wrong_arg, "  (that's df/dy)")

**Pitfall 3 — $h$ too large or too small.**
Both ends fail, differently: a fat $h$ pays curvature error; a
microscopic $h$ drowns in float rounding — in the extreme it returns
exactly 0 because the two evaluations land on the same float.

In [ ]:
exact = np.exp(2.0)                    # d/dx e^{xy} at (2, 1)

for h in (0.5, 1e-6, 1e-15):
    est = (s_fn(2.0 + h, 1.0) - s_fn(2.0 - h, 1.0)) / (2 * h)
    if h > 1e-2:
        tag = "BROKEN (curvature error)"
    elif h < 1e-12:
        tag = "BROKEN (rounding noise)"
    else:
        tag = "good (course default)"
    print(f"h = {h:<8g} estimate = {est:<22.12f} "
          f"error = {abs(est - exact):.2e}   {tag}")

### Checkpoint 7

1. A classmate claims $\partial/\partial v$ of $u v^2$ is $2v$.
   Which pitfall is this, and what is the correct partial?
2. Your `num_gradient` with $h = 10^{-18}$ returns an all-zeros gradient
   for a function you *know* is not flat.
   Explain exactly what happened and state the fix.

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. $g(3, 2) = 3 \cdot 4 - 6 = 6$; $\;g(-1, 4) = (-1)(16) - (-2) = -14$.
2. `F` has shape `(201, 301)`; the *columns* walk along $x$ (the 301-long
   axis came from `xs`).
3. The surface is steep there: the value changes by one full contour step
   over a short horizontal distance.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. $f_x = 10xy^3$, $\;f_y = 15x^2y^2$.
2. $h_s = 3s^2 + 4t \to 3 + 8 = 11$; $\;h_t = 4s - 2t \to 4 - 4 = 0$.
3. True — $y^4$ contains no $x$, so with $y$ frozen it is a constant, and
   constants have derivative 0.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. $\partial_u = 4(3u - v)^3 \cdot 3 = 12(3u - v)^3$;
   $\;\partial_v = 4(3u - v)^3 \cdot (-1) = -4(3u - v)^3$.
2. $\partial_x = 2y\, e^{2x} \to 2 \cdot 5 \cdot 1 = 10$;
   $\;\partial_y = e^{2x} \to 1$.
3. $\partial_x = y\cos(xy)$, $\;\partial_y = x\cos(xy)$ — same outer
   factor, each variable contributing its own inner derivative.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. $\nabla g = (2x + y,\; x)$; at $(1, 2)$: $(4, 1)$.
2. Every partial is 3, so $\nabla f = (3, 3, \dots, 3)$ with $d$
   entries — the gradient always has exactly one entry per input.
3. $90^\circ$ — gradient arrows are orthogonal to the contour through
   their base point (along the contour the rate is 0; straight across is
   steepest).

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. $\lVert(-6, 8)\rVert = 10$, so the steepest-increase direction is
   $(-6/10,\; 8/10) = (-0.6,\, 0.8)$, and the rate along it is $10$.
2. $(-6, 8) \cdot (1, 0) = -6$: moving east *decreases* $f$ at rate 6.
3. The two unit vectors orthogonal to the gradient:
   $(0.8,\, 0.6)$ and $(-0.8,\, -0.6)$.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. $(16 - 4)/2 = 6 = f'(3)$ exactly — on a quadratic the curvature error
   cancels perfectly, so the central difference is exact for *any* $h$
   (only rounding limits it).
2. Best $10^{-6}$ (near the valley floor); then $0.5$ (curvature error —
   $e^{xy}$ bends, unlike a quadratic); worst $10^{-13}$ (rounding noise
   from subtracting nearly identical floats).
3. Nudging one coordinate isolates that coordinate's slice — the freeze.
   Nudging all coordinates at once estimates a *directional* rate along
   the all-ones diagonal (proportional to
   $\nabla f \cdot (1, 1, \dots, 1)$), not any single partial.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. Pitfall 1's cousin: the frozen factor $u$ was dropped.
   With $u$ frozen, $u v^2$ is $(\text{constant}) \cdot v^2$, so the
   partial is $2uv$, not $2v$.
2. With $h = 10^{-18}$, `point + step` rounds to the same float as
   `point` (the nudge is below float resolution near entries of size 1),
   so every central difference is exactly 0.
   Fix: use the course default $h = 10^{-6}$, near the valley floor of
   the error curve.

</details>